# 04 - LangSmith Tracing Setup

This notebook validates LangSmith credentials, sends traced inference traffic, and builds local charts from run metadata.

It expects:
- `LANGSMITH_API_KEY` (or `LANGCHAIN_API_KEY`) in your notebook environment
- proxy endpoint reachable at `http://localhost:8000/ollama/api/chat`
- if `localhost:8000` is unavailable, run `kubectl port-forward -n llm-observability svc/langchain-demo 8000:8000` in another terminal


In [ ]:
import os
import sys
from pathlib import Path

expected = "/usr/local/bin/python3.11"
print(f"Python executable: {sys.executable}")
print(f"Python version   : {sys.version.split()[0]}")
if Path(sys.executable).resolve().as_posix() != expected:
    print(f"WARNING: This notebook is designed for {expected}.")
    print("Switch the notebook kernel to Python 3.11 before continuing.")
else:
    print("Kernel check passed.")


In [ ]:
import importlib
import subprocess
import sys

packages = ["requests", "pandas", "matplotlib", "pyyaml", "langsmith", "tabulate"]
missing = []
for pkg in packages:
    module_name = "yaml" if pkg == "pyyaml" else pkg
    try:
        importlib.import_module(module_name)
    except Exception:
        missing.append(pkg)

if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
else:
    print("All common packages are already installed.")


In [ ]:
import os
import time
from datetime import datetime, timedelta, timezone

import matplotlib.pyplot as plt
import pandas as pd
import requests
from langsmith import Client

LANGSMITH_ENDPOINT = os.getenv("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com")
LANGSMITH_PROJECT = os.getenv("LANGSMITH_PROJECT", "k3s-ollama-gemma-local")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY") or os.getenv("LANGCHAIN_API_KEY")
PROXY_HEALTH_URL = "http://localhost:8000/healthz"
PROXY_CHAT_URL = "http://localhost:8000/ollama/api/chat"
PROXY_PORT_FORWARD_CMD = "kubectl port-forward -n llm-observability svc/langchain-demo 8000:8000"
MODEL_NAME = os.getenv("OLLAMA_MODEL", "gemma3-1b-it-gguf-local")


def url_available(url: str, timeout: int = 3) -> bool:
    try:
        r = requests.get(url, timeout=timeout)
        return r.status_code < 500
    except Exception:
        return False


def find_nested_values(obj, wanted_keys: set[str]) -> list[tuple[str, object]]:
    found = []
    if isinstance(obj, dict):
        for key, value in obj.items():
            if key in wanted_keys:
                found.append((key, value))
            found.extend(find_nested_values(value, wanted_keys))
    elif isinstance(obj, (list, tuple)):
        for item in obj:
            found.extend(find_nested_values(item, wanted_keys))
    return found


def first_numeric(candidates: list[tuple[str, object]]) -> tuple[float | None, str | None]:
    for key, value in candidates:
        if isinstance(value, bool):
            continue
        if isinstance(value, (int, float)):
            return float(value), key
        if isinstance(value, str):
            try:
                return float(value), key
            except ValueError:
                continue
    return None, None


def extract_run_metrics(run) -> dict[str, object]:
    outputs = getattr(run, "outputs", {}) or {}
    status_code, status_source = first_numeric(find_nested_values(outputs, {"status_code", "http_status", "response_status"}))
    latency_ms, latency_source = first_numeric(find_nested_values(outputs, {"latency_ms", "duration_ms", "elapsed_ms"}))
    if latency_ms is None:
        start_time = getattr(run, "start_time", None)
        end_time = getattr(run, "end_time", None)
        if start_time is not None and end_time is not None:
            latency_ms = round((end_time - start_time).total_seconds() * 1000, 2)
            latency_source = "timestamps"
    if status_code is not None:
        status_code = int(status_code)
    return {
        "status_code": status_code,
        "status_source": status_source,
        "latency_ms": latency_ms,
        "latency_source": latency_source,
    }


PROXY_AVAILABLE = url_available(PROXY_HEALTH_URL)
client = None
if not LANGSMITH_API_KEY:
    print("LANGSMITH_API_KEY/LANGCHAIN_API_KEY is not set. LangSmith API cells will run in skip mode.")
else:
    try:
        client = Client(api_url=LANGSMITH_ENDPOINT, api_key=LANGSMITH_API_KEY)
    except Exception as exc:
        print(f"Unable to initialize LangSmith client: {exc}")

print("LangSmith endpoint:", LANGSMITH_ENDPOINT)
print("LangSmith project :", LANGSMITH_PROJECT)
print("PROXY_AVAILABLE   :", PROXY_AVAILABLE)
print("LANGSMITH_READY   :", client is not None)
if not PROXY_AVAILABLE:
    print("Proxy preflight failed. In another terminal run:")
    print(f"  {PROXY_PORT_FORWARD_CMD}")


## Verify Project Access


In [ ]:
project_names = []
if client is None:
    print("Skipping project listing: LangSmith client is not configured.")
else:
    try:
        projects = list(client.list_projects(limit=50))
        project_names = [p.name for p in projects]
        print(f"Visible projects: {len(project_names)}")
        print("Target project present:", LANGSMITH_PROJECT in project_names)
    except Exception as exc:
        print(f"Project listing failed: {exc}")

project_names[:15]


## Generate Traced Inference Samples via Proxy


In [ ]:
prompts = [
    "What does observability mean for LLM inference?",
    "Give a one paragraph summary of LangSmith runs.",
    "Why include status code in traces?",
    "How can I reduce memory in a local k3s stack?",
]

# Keep tracing sample generation intentionally light to avoid starving /healthz probes.
MAX_ITERS = 4
REQUEST_TIMEOUT_SECONDS = 120
SLEEP_BETWEEN_CALLS_SECONDS = 2
MAX_PREDICT = 64

sent = []
proxy_ok = PROXY_AVAILABLE
if not proxy_ok:
    print("Skipping traced proxy sample generation: proxy endpoint is not reachable on localhost:8000")
    print("In another terminal run:")
    print(f"  {PROXY_PORT_FORWARD_CMD}")
else:
    for i in range(MAX_ITERS):
        prompt = prompts[i % len(prompts)]
        body = {
            "model": MODEL_NAME,
            "stream": False,
            "messages": [{"role": "user", "content": prompt}],
            # Cap generated tokens so each traced request is shorter and safer for single-worker proxy.
            "options": {"num_predict": MAX_PREDICT},
        }

        # Re-check local proxy health before each request.
        if not url_available(PROXY_HEALTH_URL, timeout=2):
            sent.append(
                {
                    "index": i + 1,
                    "status": -1,
                    "prompt": prompt,
                    "error": "proxy /healthz is not reachable before request; stopping sample generation",
                }
            )
            proxy_ok = False
            break

        try:
            r = requests.post(PROXY_CHAT_URL, json=body, timeout=REQUEST_TIMEOUT_SECONDS)
            sent.append({"index": i + 1, "status": r.status_code, "prompt": prompt})
        except Exception as exc:
            msg = str(exc)
            sent.append({"index": i + 1, "status": -1, "prompt": prompt, "error": msg})
            if "Connection aborted" in msg or "Connection refused" in msg:
                print("Proxy connection dropped; stopping further traced sample requests.")
                proxy_ok = False
                break
        finally:
            time.sleep(SLEEP_BETWEEN_CALLS_SECONDS)

pd.DataFrame(sent)


## Pull Recent Runs From LangSmith


In [ ]:
runs_df = pd.DataFrame(
    columns=[
        "run_id",
        "name",
        "run_type",
        "start_time",
        "error",
        "status_code",
        "status_source",
        "latency_ms",
        "latency_source",
        "tags",
    ]
)

if client is None:
    print("Skipping run fetch: LangSmith client is not configured.")
else:
    try:
        cutoff = datetime.now(timezone.utc) - timedelta(hours=2)
        runs = list(client.list_runs(project_name=LANGSMITH_PROJECT, start_time=cutoff, limit=200))
        print("Runs fetched:", len(runs))

        rows = []
        for run in runs:
            metrics = extract_run_metrics(run)
            rows.append(
                {
                    "run_id": str(getattr(run, "id", "")),
                    "name": getattr(run, "name", ""),
                    "run_type": getattr(run, "run_type", ""),
                    "start_time": getattr(run, "start_time", None),
                    "error": getattr(run, "error", None),
                    "status_code": metrics["status_code"],
                    "status_source": metrics["status_source"],
                    "latency_ms": metrics["latency_ms"],
                    "latency_source": metrics["latency_source"],
                    "tags": ",".join(getattr(run, "tags", []) or []),
                }
            )

        runs_df = pd.DataFrame(rows)
    except Exception as exc:
        print(f"Run fetch failed: {exc}")

runs_df.head(20)


In [ ]:
if not runs_df.empty:
    runs_df["is_error"] = runs_df["error"].notna()
    runs_df["latency_ms"] = pd.to_numeric(runs_df["latency_ms"], errors="coerce")
    runs_df["start_time"] = pd.to_datetime(runs_df["start_time"], errors="coerce")

    print("Error runs:", int(runs_df["is_error"].sum()))
    print("Latency rows:", int(runs_df["latency_ms"].notna().sum()))
    print("Latency source counts:")
    display(runs_df["latency_source"].fillna("missing").value_counts().rename_axis("latency_source").to_frame("runs"))

    display(runs_df[["start_time", "name", "status_code", "status_source", "latency_ms", "latency_source", "error"]].tail(20))

    chart_df = runs_df[runs_df["latency_ms"].notna()].copy()
    chart_df = chart_df.sort_values("start_time").tail(50)

    if not chart_df.empty:
        ax = chart_df.plot(
            x="start_time",
            y="latency_ms",
            figsize=(10, 4),
            marker="o",
            title="LangSmith Latency (Recent Runs)",
        )
        ax.set_ylabel("Latency (ms)")
        plt.show()
    else:
        print("No numeric latency found in LangSmith outputs or run timestamps.")


In [ ]:
# Quick tag breakdown to validate open-webui/proxy activity
if runs_df.empty:
    print("No runs available for tag breakdown.")
else:
    tag_counts = (
        runs_df["tags"]
        .fillna("")
        .str.split(",")
        .explode()
        .str.strip()
    )
    tag_counts = tag_counts[tag_counts != ""]
    print(tag_counts.value_counts().head(20))



## Done
Open `smith.langchain.com`, choose project **`k3s-ollama-gemma-local`**, and compare dashboard charts with the notebook output.
